In [1]:
import os
from mr_eval.utils.utils import *
from tqdm import tqdm
import random
random.seed(42)

# refocus_data_path = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/ReFocus_data/train_chartQA_h_bar_wbb/gpt-4o-2024-08-06-mix_orig_build_train_nogold"
refocus_data_path = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/ReFocus_data/train_chartQA_v_bar_wbb/gpt-4o-2024-08-06-mix_orig_build_train_nogold"
# refocus_data_path = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/ReFocus_data/train_chartQA_h_bar_wbb-nogold_errors/gpt-4o-2024-08-06-mix_orig_build_train"
# refocus_data_path = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/ReFocus_data/train_chartQA_v_bar_wbb-nogold_errors/gpt-4o-2024-08-06-mix_orig_build_train"


In [2]:
question_ids = os.listdir(refocus_data_path)
length_dict = {}
qualified_ids = []

for question_id in tqdm(question_ids):
    
    current_instance_path = os.path.join(refocus_data_path, question_id)
    current_answer_json = os.path.join(current_instance_path, "output_edit.json")
    
    conversation_data = load_json_file(current_answer_json)
    conversation_len = len(conversation_data)
    if conversation_len not in length_dict:
        length_dict[conversation_len] = 0
    length_dict[conversation_len] += 1
    if conversation_len > 2:
        qualified_ids.append(question_id)
    
    
random.shuffle(qualified_ids)


100%|██████████| 10069/10069 [00:28<00:00, 350.51it/s]


In [ ]:
qualified_ids_dict = {"qualified_ids": qualified_ids}
write_json_file(qualified_ids_dict, os.path.join("/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data", "qualified_ids.json"))



In [3]:
qualified_ids_path = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/qualified_ids.json"
output_path = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/refocus_chartqa_v_bar_wbb_candidates.jsonl"
output_path2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/refocus/data/refocus_chartqa_v_bar_wbb_selfbar_candidates.jsonl"
qualified_ids = load_json_file(qualified_ids_path)["qualified_ids"]
k=200

candidiate_ids = qualified_ids[:k]
selfbar_candidate_ids = qualified_ids[k:2*k]
candidiate_ids.extend(qualified_ids[2*k:3*k])
selfbar_candidate_ids.extend(qualified_ids[3*k:4*k])
# candidiate_ids = selfbar_candidate_ids
# output_path = output_path2

In [4]:
candidates = []
for candidate_id in candidiate_ids:
    current_instance_path = os.path.join(refocus_data_path, candidate_id)
    current_answer_json = os.path.join(current_instance_path, "output_edit.json")
    
    conversation_data = load_json_file(current_answer_json)
    conversation_data[0]["content"] = conversation_data[0]["content"][1:]
    message_list = conversation_data
    
    example_data = load_json_file(os.path.join(current_instance_path, "example.json"))
    qid = candidate_id
    res = dict(qid = qid, message_list = message_list, extra_info = example_data)
    candidates.append(res)

write_jsonl(candidates, output_path)

In [4]:
selfbar_candidates = []
for candidate_id in selfbar_candidate_ids:
    current_instance_path = os.path.join(refocus_data_path, candidate_id)
    current_answer_json = os.path.join(current_instance_path, "output_edit.json")
    
    conversation_data = load_json_file(current_answer_json)
    conversation_data[0]["content"] = conversation_data[0]["content"][1:]
    message_list = conversation_data
    
    example_data = load_json_file(os.path.join(current_instance_path, "example.json"))
    qid = candidate_id
    res = dict(qid = qid, message_list = message_list, extra_info = example_data)
    selfbar_candidates.append(res)

write_jsonl(selfbar_candidates, output_path2)

In [5]:
len(selfbar_candidates)

400